In [1]:
# ------------------------------------------------------------
# Cell 1 — Setup / Imports / Paths
# ------------------------------------------------------------

from pathlib import Path
import sys
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path.cwd()

# Make sure local modules in this folder are importable
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import flow_data_utils as fdu

CACHE_PATH = ROOT / "cache" / "tributary_cache.joblib"
PLOTS_DIR = ROOT / "Plots_Discharge"
DATA_DIR = ROOT / "Data_Filled"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

REFIT_STATIONS = ["07381000", "08012470", "02470629"]
SHORT_GAP_DAYS = 3
BAYOU_LAFOURCHE_CHANGE_DATE = "2016-01-01"


In [2]:
# ------------------------------------------------------------
# Cell 2 — Load cached processed data
# ------------------------------------------------------------

if not CACHE_PATH.exists():
    raise FileNotFoundError(f"Cache not found: {CACHE_PATH}")

objs = joblib.load(CACHE_PATH)

station_metadata = objs.get("station_metadata", {}) or {}
station_names = objs.get("station_names", {}) or {}
flow_data_reindexed = objs.get("flow_data_reindexed", {}) or {}
filled_flow_data = objs.get("filled_flow_data", {}) or {}


In [6]:
# Data, save via fdu.save_to_csv
import pandas as pd
from pathlib import Path

sid = "07381000"
change_date = pd.Timestamp(BAYOU_LAFOURCHE_CHANGE_DATE)

# slice by datetime index (no reset_index to avoid level_0 collisions)
df = flow_data_reindexed[sid].copy()
pre_df = df.loc[df.index < change_date].copy()
post_df = df.loc[df.index >= change_date].copy()

# drop index-like columns that can conflict
for d in (pre_df, post_df):
    for c in ["level_0", "index"]:
        if c in d.columns:
            d.drop(columns=[c], inplace=True)

# run the pipeline (adjust function name/args if yours differ)
filled_pre = fdu.gap_filling_pipeline_with_metadata(pre_df, station_metadata.get(sid, {}).copy(),
                                                    station_names.get(sid, sid) + " (pre-2016)",
                                                    flow_data_reindexed, discharge_col="Discharge", stage_col="Stage")
filled_post = fdu.gap_filling_pipeline_with_metadata(post_df, station_metadata.get(sid, {}).copy(),
                                                     station_names.get(sid, sid) + " (post-2016)",
                                                     flow_data_reindexed, discharge_col="Discharge", stage_col="Stage")

# Save using helper function 
outdir = Path(DATA_DIR)
out_pre = outdir / "07381000_BayouLafourche_pre2016_filled.csv"
out_post = outdir / "07381000_BayouLafourche_post2016_filled.csv"
fdu.save_to_csv(filled_pre, str(out_pre))
fdu.save_to_csv(filled_post, str(out_post))
print("Saved:", out_pre, out_post)

Saving columns: ['Date', 'Discharge (m3/s)', 'Discharge_filled_method'] to C:\Users\p00278065\Coastal Master Plan 2029\ICM_pre-processing\CMP_ICM_pre-processing_CLONE\ICM_pre-processing\observed_tributary_flows\Data_Filled\07381000_BayouLafourche_pre2016_filled.csv
Saving columns: ['Date', 'Discharge (m3/s)', 'Discharge_filled_method'] to C:\Users\p00278065\Coastal Master Plan 2029\ICM_pre-processing\CMP_ICM_pre-processing_CLONE\ICM_pre-processing\observed_tributary_flows\Data_Filled\07381000_BayouLafourche_post2016_filled.csv
Saved: C:\Users\p00278065\Coastal Master Plan 2029\ICM_pre-processing\CMP_ICM_pre-processing_CLONE\ICM_pre-processing\observed_tributary_flows\Data_Filled\07381000_BayouLafourche_pre2016_filled.csv C:\Users\p00278065\Coastal Master Plan 2029\ICM_pre-processing\CMP_ICM_pre-processing_CLONE\ICM_pre-processing\observed_tributary_flows\Data_Filled\07381000_BayouLafourche_post2016_filled.csv


In [9]:
# Bayou Lafourche 
# Cell 4 — combine pre/post, save combined CSV with "_with_rev"

sid = "07381000"
name_raw = station_names.get(sid, sid)  # e.g. "Bayou Lafourche at Thibodeaux, LA"
outdir = Path(DATA_DIR)
outdir.mkdir(parents=True, exist_ok=True)

def clean_name(s):
    s = str(s or "")
    s = s.replace(",", "") \
         .replace("/", "_") \
         .replace("(", "") \
         .replace(")", "")
    s = re.sub(r"\s+", "_", s.strip())
    s = re.sub(r"[^\w\-_]+", "", s)
    s = re.sub(r"_+", "_", s)
    return s.strip("_")

base = f"{clean_name(name_raw)}_{sid}_filled"
csv_path = outdir / f"{base}_with_rev.csv"
fig_path = outdir / f"{base}_H_vs_Q_with_rev.png"

def _prep_minimal(df, period_label):
    df2 = df.copy()
    # ensure Date as column
    if "Date" not in df2.columns and pd.api.types.is_datetime64_any_dtype(df2.index):
        df2 = df2.reset_index()
        if df2.columns[0] != "Date":
            df2 = df2.rename(columns={df2.columns[0]: "Date"})
    # drop index-like cols
    for c in ["level_0", "index"]:
        if c in df2.columns:
            df2 = df2.drop(columns=[c])
    # ensure discharge column has units header
    if "Discharge (m3/s)" not in df2.columns and "Discharge" in df2.columns:
        df2 = df2.rename(columns={"Discharge": "Discharge (m3/s)"})
    # keep only minimal columns (plus period)
    keep_cols = [c for c in ["Date", "Discharge (m3/s)", "Discharge_filled_method", "Stage"] if c in df2.columns]
    df_min = df2.loc[:, keep_cols].copy()
    df_min["period"] = period_label
    return df_min

# Prepare both pieces
pre_min = _prep_minimal(filled_pre, "pre-2016")
post_min = _prep_minimal(filled_post, "post-2016")

# Combine and sort
combined = pd.concat([pre_min, post_min], ignore_index=True, sort=False)
# normalize Date column to dates only (if datetime)
combined["Date"] = pd.to_datetime(combined["Date"], errors="coerce").dt.normalize()

# Save only the compact 3 columns + period (excluding Stage unless present)
cols_to_write = [c for c in ["Date", "Discharge (m3/s)", "Discharge_filled_method", "period"] if c in combined.columns]
combined.to_csv(csv_path, index=False, columns=cols_to_write, encoding="utf-8")
print("Saved combined CSV:", csv_path)


Saved combined CSV: C:\Users\p00278065\Coastal Master Plan 2029\ICM_pre-processing\CMP_ICM_pre-processing_CLONE\ICM_pre-processing\observed_tributary_flows\Data_Filled\Bayou_Lafourche_at_Thibodeaux_LA_07381000_filled_with_rev.csv


In [23]:
# Quick tests for filled_pre / filled_post before saving
import pandas as pd
from pathlib import Path

def test_filled(df, label="df", change_date=None):
    info = {"label": label, "ok": True, "messages": []}

    # locate date
    if "Date" in df.columns:
        dates = pd.to_datetime(df["Date"], errors="coerce")
    elif pd.api.types.is_datetime64_any_dtype(df.index):
        dates = pd.to_datetime(df.index)
    else:
        info["ok"] = False
        info["messages"].append("No Date column and index is not datetime-like.")
        return info

    if dates.isna().any():
        info["messages"].append(f"Warning: {dates.isna().sum()} unparseable dates.")

    info["n_rows"] = len(df)
    info["date_min"] = dates.min()
    info["date_max"] = dates.max()
    if change_date is not None:
        info["contains_change_date"] = (pd.to_datetime(change_date) >= info["date_min"]) and (pd.to_datetime(change_date) <= info["date_max"])

    # find flow column
    flow_col = None
    if "Discharge (m3/s)" in df.columns:
        flow_col = "Discharge (m3/s)"
        unit_hint = "m3/s (explicit)"
    elif "Discharge" in df.columns:
        flow_col = "Discharge"
        unit_hint = "unknown (named 'Discharge')"
    else:
        for c in df.columns:
            lc = str(c).lower()
            if "flow" in lc or "discharge" in lc:
                flow_col = c
                unit_hint = "unknown (detected)"
                break

    if flow_col is None:
        info["ok"] = False
        info["messages"].append("No flow/discharge column detected.")
        return info

    ser = pd.to_numeric(df[flow_col], errors="coerce")
    info.update({
        "flow_col": flow_col,
        "flow_count_nonnull": int(ser.notna().sum()),
        "flow_count_null": int(ser.isna().sum()),
        "flow_mean": float(ser.mean(skipna=True)) if ser.notna().any() else None,
        "flow_median": float(ser.median(skipna=True)) if ser.notna().any() else None,
        "flow_min": float(ser.min(skipna=True)) if ser.notna().any() else None,
        "flow_max": float(ser.max(skipna=True)) if ser.notna().any() else None,
        "unit_hint": unit_hint
    })

    # simple unit heuristic
    mean_val = info["flow_mean"]
    if mean_val is None:
        info["messages"].append("Flow column contains no numeric values.")
    else:
        # heuristic: very large values indicate CFS (adjust threshold if needed)
        if flow_col == "Discharge (m3/s)" or mean_val < 50:
            info["messages"].append(f"Flow values plausibly in m3/s (mean={mean_val:.2f}).")
        elif mean_val >= 100:
            info["messages"].append(f"Flow mean is large ({mean_val:.1f}) — values may be in CFS. Consider converting to m3/s before saving.")
            info["messages"].append("Suggested conversion: df['Discharge'] = pd.to_numeric(df[flow_col], errors='coerce') * 0.0283168")
        else:
            info["messages"].append(f"Flow mean is {mean_val:.2f} — unclear units; inspect a sample.")

    # percent missing
    pct_na = 100.0 * info["flow_count_null"] / max(1, info["n_rows"])
    info["flow_pct_na"] = pct_na
    if pct_na > 20:
        info["messages"].append(f"Warning: {pct_na:.1f}% of flow values are missing.")

    # sample rows to inspect
    info["sample_head"] = df.head(3)
    info["sample_tail"] = df.tail(3)

    return info

# Run tests (adjust change_date if needed)
change_date = "2016-01-01"
results_pre = test_filled(filled_pre, label="filled_pre", change_date=change_date)
results_post = test_filled(filled_post, label="filled_post", change_date=change_date)

# Pretty print a concise summary
def summarize(res):
    print(f"--- {res['label']} ---")
    print("Rows:", res.get("n_rows"))
    print("Date range:", res.get("date_min"), "->", res.get("date_max"))
    print("Flow column:", res.get("flow_col"), "/", res.get("unit_hint"))
    print("Non-null flow:", res.get("flow_count_nonnull"), "Null flow:", res.get("flow_count_null"), f"({res.get('flow_pct_na'):.1f}% missing)")
    if res.get("flow_mean") is not None:
        print("Flow stats: mean={:.2f}, median={:.2f}, min={:.2f}, max={:.2f}".format(
            res["flow_mean"], res["flow_median"], res["flow_min"], res["flow_max"]))
    for m in res.get("messages", []):
        print("-", m)
    print("\nSample head:")
    print(res["sample_head"])
    print("\nSample tail:")
    print(res["sample_tail"])
    print("\n")

summarize(results_pre)
summarize(results_post)

# If both seem OK, optionally save (uncomment to run)
# prepare_and_call_save(filled_pre, Path(DATA_DIR)/"07381000_BayouLafourche_pre2016_filled.csv")
# prepare_and_call_save(filled_post, Path(DATA_DIR)/"07381000_BayouLafourche_post2016_filled.csv")

--- filled_pre ---
Rows: 3652
Date range: 2006-01-01 00:00:00 -> 2015-12-31 00:00:00
Flow column: Discharge / unknown (named 'Discharge')
Non-null flow: 3652 Null flow: 0 (0.0% missing)
Flow stats: mean=5.78, median=5.37, min=0.93, max=25.03
- Flow values plausibly in m3/s (mean=5.78).

Sample head:
   level_0  index       Date  Discharge     Stage   Station Discharge_source  \
0        0      0 2006-01-01   5.550093  1.072896  07381000            daily   
1        1      1 2006-01-02   5.521776  1.078992  07381000            daily   
2        2      2 2006-01-03   5.550093  1.075944  07381000            daily   

  Stage_source Discharge_filled_method RC_donor_station RC_donor_q_source  \
0        daily                original                                      
1        daily                original                                      
2        daily                original                                      

  RC_donor_stage_source RC_stage_source  
0                          

In [22]:
# Minimal fix: drop index-like conflicting columns, run pipeline, save outputs
import re
from pathlib import Path
sid = "07381000"
change_date = pd.Timestamp(BAYOU_LAFOURCHE_CHANGE_DATE)

df = flow_data_reindexed[sid].copy()
pre_df = df.loc[df.index < change_date].copy()
post_df = df.loc[df.index >= change_date].copy()

def _drop_conflicts(d):
    conflicts = [c for c in d.columns if re.match(r'^(level_\d+|Unnamed:\s*\d+|index)$', c)]
    if conflicts:
        d = d.drop(columns=conflicts)
    return d

pre_df = _drop_conflicts(pre_df)
post_df = _drop_conflicts(post_df)

# run the existing pipeline
meta = station_metadata.get(sid, {}).copy()
name = station_names.get(sid, sid)
filled_pre = fdu.gap_filling_pipeline_with_metadata(pre_df, meta.copy(), name + " (pre-2016)", flow_data_reindexed,
                                                   discharge_col="Discharge", stage_col="Stage")
filled_post = fdu.gap_filling_pipeline_with_metadata(post_df, meta.copy(), name + " (post-2016)", flow_data_reindexed,
                                                    discharge_col="Discharge", stage_col="Stage")

# save
out_pre = Path(DATA_DIR) / f"{sid}_{(fdu.clean_name(name) if hasattr(fdu,'clean_name') else name)}_pre2016_filled.csv"
out_post = Path(DATA_DIR) / f"{sid}_{(fdu.clean_name(name) if hasattr(fdu,'clean_name') else name)}_post2016_filled.csv"
fdu.save_to_csv(filled_pre, str(out_pre))
fdu.save_to_csv(filled_post, str(out_post))

print("Saved:", out_pre, out_post)

AttributeError: module 'flow_data_utils' has no attribute 'save_to_csv'

In [19]:
#Bayou Lafourche

sid = REFIT_STATIONS[0]  # expects "07381000" to be in REFIT_STATIONS
change_date = pd.Timestamp(BAYOU_LAFOURCHE_CHANGE_DATE)

df = flow_data_reindexed[sid].copy()
df["Date"] = pd.to_datetime(df["Date"])
pre_df = df[df["Date"] < change_date].copy()
post_df = df[df["Date"] >= change_date].copy()

station_metadata.setdefault(sid, {}).setdefault("piecewise_info", {})["break_date"] = str(change_date.date())

print(f"{sid} split: pre_rows={len(pre_df)}, post_rows={len(post_df)}")

pre_df, post_df

KeyError: 'Date'

In [12]:
# ------------------------------------------------------------
# Cell 3 — Shared helpers for station refit cells
# ------------------------------------------------------------

def clean_name(s: str) -> str:
    return (s or "").replace(" ", "_").replace(",", "").replace("/", "_")

def run_station_gapfill(station_id: str):
    """
    Run the shared gap-filling pipeline for one station using cached data.
    """
    if station_id not in flow_data_reindexed:
        raise KeyError(f"{station_id}: not found in flow_data_reindexed.")
    if station_id not in station_metadata:
        raise KeyError(f"{station_id}: not found in station_metadata.")

    df = flow_data_reindexed[station_id]
    meta = station_metadata[station_id].copy()
    name = station_names.get(station_id, station_id)

    filled_df = fdu.gap_filling_pipeline_with_metadata(
        df,
        meta,
        name,
        flow_data_reindexed,
        discharge_col="Discharge",
        stage_col="Stage",
    )
    return filled_df, meta, name

In [6]:
# ------------------------------------------------------------
# Load cached objects and read in data
# ------------------------------------------------------------

# Load the cache produced by your main workflow
if not CACHE_PATH.exists():
    raise FileNotFoundError(
        f"Cache not found: {CACHE_PATH}\n"
        "Expected ./cache/tributary_cache.joblib relative to your notebook run folder."
    )

objs = joblib.load(CACHE_PATH)

# Core datasets used throughout the notebook
# Station metadata (rating curve strings, donor station IDs, etc.)
station_metadata = objs.get("station_metadata", {}) or {}

# Station ID -> nice name
station_names = objs.get("station_names", {}) or {}

# Reindexed time series (usually includes 'Discharge', 'Stage', and maybe methods)
flow_data_reindexed = objs.get("flow_data_reindexed", {}) or {}

# Filled time series (typically includes filled discharge + method tags)
filled_flow_data = objs.get("filled_flow_data", {}) or {}


In [7]:
# ------------------------------------------------------------
# Utilities 
# ------------------------------------------------------------

def clean_name(s: str) -> str:
    """Simple filename-safe station name."""
    return (s or "").replace(" ", "_").replace(",", "").replace("/", "_")

def as_daily_index(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure dataframe is indexed by normalized daily datetime.
    Accepts either a 'Date' column or an existing datetime-like index.
    """
    df = df.copy()
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
        df = df.set_index("Date")
    else:
        df.index = pd.to_datetime(df.index, errors="coerce").normalize()
    return df.sort_index()

def daily_mean(series: pd.Series) -> pd.Series:
    """Convert to numeric, normalize index to day, and take daily mean."""
    s = pd.to_numeric(series, errors="coerce")
    s.index = pd.to_datetime(s.index, errors="coerce").normalize()
    return s.groupby(level=0).mean().sort_index()

def find_nan_runs(is_nan: np.ndarray):
    """Return list of (start, end) pairs for consecutive True runs in a boolean array."""
    n = len(is_nan)
    i = 0
    runs = []
    while i < n:
        if is_nan[i]:
            start = i
            while i < n and is_nan[i]:
                i += 1
            runs.append((start, i))
        else:
            i += 1
    return runs

def rc_kind(rc_str: str) -> str:
    """
    Classify RC strings by whether they reference H/h and/or Q.
    Returns: 'H', 'Q', 'HQ', or 'unknown'
    """
    s = (rc_str or "").strip()
    has_q = re.search(r"\bQ\b", s) is not None
    has_h = re.search(r"\bH\b", s) is not None or re.search(r"\bh\b", s) is not None
    if has_h and not has_q:
        return "H"
    if has_q and not has_h:
        return "Q"
    if has_h and has_q:
        return "HQ"
    return "unknown"

def eval_rc_scalar(rc_str: str, x: float) -> float:
    """
    Evaluate a rating curve string at scalar x.

    NOTE:
      - Intended for simple H->Q or Q->Q curves.
      - We bind H/h and Q to the same scalar x so either variable name works.
      - If you later support true HQ formulas, we can add an evaluator that accepts both H and Q.
    """
    if not rc_str or pd.isna(x):
        return np.nan
    try:
        local = {"np": np, "H": x, "h": x, "Q": x}
        return float(eval(rc_str, {"__builtins__": {}}, local))
    except Exception:
        return np.nan

In [8]:
# ------------------------------------------------------------
# Create consistent plots
# ------------------------------------------------------------

# Plot colors (match your prior convention)
PLOT_COLORS = {
    "Original": "#000000",         # black
    "Interpolated": "#00bcd4",     # cyan
    "Rating Curve": "#f39c12",     # orange
    "Long Gap Interp": "#6a1b9a",  # purple
    "Unknown": "#7f7f7f",
}

def normalize_method_for_plot(method: str) -> str:
    """Map internal method tags to a small set of plot categories."""
    m = (method or "").strip().lower()
    if m == "original" or m == "":
        return "Original"
    if m in ("interpolated", "interp_short"):
        return "Interpolated"
    if m in ("interpolated_long_gap", "long_gap_interp"):
        return "Long Gap Interp"
    if m in ("rating_curve", "rating_curve_long_gap", "rc"):
        return "Rating Curve"
    return "Unknown"

def plot_discharge_timeseries(df_out: pd.DataFrame, station_title: str, png_out: Path):
    """
    Scatter plot of daily discharge colored by fill method.
    Expects columns: Date, Discharge (m3/s), Discharge_filled_method
    """
    dfp = df_out.copy()
    dfp["Date"] = pd.to_datetime(dfp["Date"], errors="coerce")
    cat = dfp["Discharge_filled_method"].astype(str).map(normalize_method_for_plot)

    fig, ax = plt.subplots(figsize=(16, 6))

    # Plot originals first (smaller markers)
    m0 = cat.eq("Original")
    if m0.any():
        ax.scatter(
            dfp.loc[m0, "Date"], dfp.loc[m0, "Discharge (m3/s)"],
            s=12, c=PLOT_COLORS["Original"], label="Original",
            linewidths=0, alpha=0.95
        )

    # Plot fill categories on top (bigger markers)
    for label in ["Interpolated", "Rating Curve", "Long Gap Interp"]:
        m = cat.eq(label)
        if m.any():
            ax.scatter(
                dfp.loc[m, "Date"], dfp.loc[m, "Discharge (m3/s)"],
                s=24, c=PLOT_COLORS[label], label=label,
                linewidths=0, alpha=0.95
            )

    ax.set_title(station_title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Discharge (m3/s)")
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)

    ax.xaxis.set_major_locator(mdates.YearLocator(base=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.legend(loc="upper right", frameon=True)

    fig.savefig(png_out, dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)
    print("Saved plot:", png_out)

In [ ]:
# ------------------------------------------------------------
# Cell 2: Imports + load cached objects needed for refit/review
# ------------------------------------------------------------

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ---------------------- Load cache ----------------------
if not CACHE_PATH.exists():
    raise FileNotFoundError(
        f"Cache not found at: {CACHE_PATH}\n"
        "Expected ./cache/tributary_cache.joblib relative to your notebook run folder."
    )

objs = joblib.load(CACHE_PATH)
print("Loaded cache keys:", list(objs.keys()))

# ---------------------- Pull commonly used objects ----------------------
# Station metadata table (RC strings, donor station IDs, etc.)
station_metadata = objs.get("station_metadata", {}) or {}

# Station name mapping
station_names = objs.get("station_names", {}) or {}

# Time series data (expected to include Date/Discharge/Stage and filled method columns)
flow_data_reindexed = objs.get("flow_data_reindexed", {}) or {}
filled_flow_data = objs.get("filled_flow_data", {}) or {}


In [2]:
# ------------------------------------------------------------
# Cell 3: Small utilities (naming, daily indexing, RC evaluation, gap finding)
# Assumes imports + cache objects are already loaded in Cell 2.
# ------------------------------------------------------------

import re  # used for rating curve string classification

def _clean_name(s: str) -> str:
    """Filename-safe station name (simple)."""
    return (s or "").replace(" ", "_").replace(",", "").replace("/", "_")

def _as_daily_index(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure dataframe is indexed by normalized daily datetime.
    Accepts either a 'Date' column or an existing datetime-like index.
    """
    df = df.copy()
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
        df = df.set_index("Date")
    else:
        df.index = pd.to_datetime(df.index, errors="coerce").normalize()
    return df.sort_index()

def _daily_mean(series: pd.Series) -> pd.Series:
    """Convert to numeric, normalize index to day, and take daily mean."""
    s = pd.to_numeric(series, errors="coerce")
    s.index = pd.to_datetime(s.index, errors="coerce").normalize()
    return s.groupby(level=0).mean().sort_index()

def _find_nan_runs(is_nan: np.ndarray):
    """Return list of (start, end) pairs for consecutive True runs in a boolean array."""
    n = len(is_nan)
    i = 0
    runs = []
    while i < n:
        if is_nan[i]:
            start = i
            while i < n and is_nan[i]:
                i += 1
            runs.append((start, i))
        else:
            i += 1
    return runs

def rc_kind(rc_str: str) -> str:
    """
    Classify RC strings by whether they reference H/h and/or Q.
    Returns: 'H', 'Q', 'HQ', or 'unknown'
    """
    s = (rc_str or "").strip()
    has_q = re.search(r"\bQ\b", s) is not None
    has_h = re.search(r"\bH\b", s) is not None or re.search(r"\bh\b", s) is not None
    if has_h and not has_q:
        return "H"
    if has_q and not has_h:
        return "Q"
    if has_h and has_q:
        return "HQ"
    return "unknown"

def eval_rc_scalar(rc_str: str, x: float) -> float:
    """
    Evaluate a rating curve string at scalar x.

    NOTE:
      - This is intended for simple H->Q or Q->Q curves.
      - We bind H/h and Q to the same scalar x on purpose so either variable name works.
      - If you later support true HQ formulas, we’ll add an eval function that accepts both H and Q.
    """
    if not rc_str or pd.isna(x):
        return np.nan
    try:
        local = {"np": np, "H": x, "h": x, "Q": x}
        return float(eval(rc_str, {"__builtins__": {}}, local))
    except Exception:
        return np.nan

07381000: quick-fit post-2016 RC: 6.40481*H + 4.34788 (N=3421)
Saved CSV: C:\Users\p00278065\Coastal Master Plan 2029\Tributary-Flows_2029\data_csv\Revised_Rating_Curves\Bayou_Lafourche_at_Thibodeaux_LA_07381000_filled_rev.csv
Saved plot: C:\Users\p00278065\Coastal Master Plan 2029\Tributary-Flows_2029\plots\Revised_Rating_Curves\Bayou_Lafourche_at_Thibodeaux_LA_07381000_Timeseries_Revised_rev.png
